![Week 4 Day 4](../assets/banners/w4d4.png)

# Week 4 Day 4 - Faithful Transcription: Never "Fix" the Student

Yesterday you turned answer sheets into clean, right-sized page images, and packed one into a multimodal message. You even caught a glimpse of the system prompt waiting at the top of it.

Today that prompt gets the spotlight, because it carries the most important rule of the week: **GradeWise copies, it never edits.** If Sam writes *"semafore"*, the transcript says *"semafore"*. If he gets deadlock wrong, the transcript gets it wrong in exactly the same way. A grader can only be fair to what the student *actually wrote*.

Then we do the unglamorous work that makes transcription usable: matching every scrawled label ("4(b)", "Ans 4 ii", "contd.") to an exam question, spotting missing answers, and putting a **number** on how much to trust each one.

![Week 4 roadmap](../assets/roadmap/week4.png)

### 🎯 By the end of this lab you will
1. 🔬 Dissect GradeWise's verbatim-transcription prompt, rule by rule
2. 📏 Use **CER** and **WER** on the real ground truth to prove that "fixing" spelling is a transcription error, and an unfair one
3. 🧭 Map messy labels and multi-page answers to question IDs with plain, testable Python
4. 🎯 Combine self-reported confidence, `[illegible]` markers and a **second reading** into one score that decides what a human checks

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 4 - Faithful Transcription with a Vision LLM (Never "Fix" the Student) | 🧪 |
| 2 | Day 4 - Mapping Answers to Questions: Labels, Continuations, Missing Answers | 🧪 |
| 3 | Day 4 - Confidence Scores, Illegible Flags and Double-Pass Verification | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Only one cell today calls a real vision model (Part 3). Everything else, including all the CER maths, the mapper and the confidence checks, is plain Python or uses a <b>fake vision model</b> that replays canned answers, just like <code>tests/test_transcription.py</code>. So it's free, and the results are the same on every machine.</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Most of today's cells still run offline; the few that call a model will")
    print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

In [ ]:
import json

from gradewise.config import DATA_DIR, OUTPUT_DIR, REPO_ROOT
from gradewise.console import success, warn

SYNTHETIC = DATA_DIR / "answer-sheets" / "synthetic"
SHEETS = {p.stem: p for p in sorted(SYNTHETIC.glob("student_*.pdf"))}
TRUTH_FILE = SYNTHETIC / "transcripts.json"
TRUTH = json.loads(TRUTH_FILE.read_text(encoding="utf-8")) if TRUTH_FILE.exists() else {}

if SHEETS and TRUTH:
    success("Synthetic answer sheets found", {"Folder": str(SYNTHETIC.relative_to(REPO_ROOT)), "Sheets": len(SHEETS)})
else:
    # ⚠️ Plan B: the sheets aren't there yet (run `uv run scripts/make_synthetic_sheets.py`, or `git pull`).
    #    We draw a small stand-in sheet so the rest of the lab still runs. Its text is our "ground truth".
    from PIL import Image, ImageDraw, ImageFont

    warn("data/answer-sheets/synthetic/ is missing or incomplete: using a small stand-in sheet instead.")
    standin = {
        "Q3": "Dedlock is when a process waits for a long time. Conditions: mutual exclution and circuler wait.",
        "Q4a": "When two threads use the same varible at the same time.",
    }
    try:
        font = ImageFont.truetype("C:/Windows/Fonts/Inkfree.ttf", 34)
    except OSError:
        font = ImageFont.load_default()
    pages = []
    for label, qid in [("3)", "Q3"), ("4a)", "Q4a")]:
        img = Image.new("RGB", (1240, 1754), "white")
        draw = ImageDraw.Draw(img)
        words, lines, line = f"{label} {standin[qid]}".split(), [], ""
        for word in words:
            if len(line) + len(word) > 48:
                lines.append(line)
                line = ""
            line += word + " "
        for i, text in enumerate([*lines, line]):
            draw.text((90, 140 + i * 70), text, fill=(28, 45, 140), font=font)
        pages.append(img)
    target = OUTPUT_DIR / "week4" / "student_standin.pdf"
    target.parent.mkdir(parents=True, exist_ok=True)
    pages[0].save(target, "PDF", resolution=150, save_all=True, append_images=pages[1:])
    SHEETS, TRUTH = {"student_standin": target}, {"student_standin": standin}

In [ ]:
from gradewise.schemas import ExamPaper

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
exam = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTION_IDS = [q.id for q in exam.questions]
print(f"{exam.title}: {len(QUESTION_IDS)} questions → {', '.join(QUESTION_IDS)}")

In [ ]:
def show_table(rows: list[dict]) -> None:
    """Render a list of dicts as a Markdown table (a tiny stand-in for pandas: no extra dependencies)."""
    if not rows:
        print("(no rows)")
        return
    cols = list(rows[0])

    def cell(value) -> str:
        return str(value).replace("|", "\\|").replace("\n", " ⏎ ")

    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(cell(r.get(c, "")) for c in cols) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

## Part 1: The verbatim-transcription prompt, dissected

Here is the system prompt every page is sent with. Read it slowly: every rule exists because a vision model, left alone, "helpfully" does the opposite.

In [ ]:
from gradewise.transcription.vision import PAGE_PROMPT, SYSTEM_PROMPT

print(SYSTEM_PROMPT)

| Rule | The failure it prevents |
|---|---|
| 1. **Verbatim** | The model fixes spelling and grammar, completes half-finished sentences, "improves" wrong answers |
| 2. **Answer labels**, as written | The model renumbers answers or silently merges two of them |
| 3. **Continuations** | An answer that runs onto page 2 gets attached to the wrong question, or lost |
| 4. **Crossed-out text** kept separate | The student's rejected first attempt gets marked |
| 5. **`[illegible]`, never guess** | The model invents plausible words for a scribble, and the grader marks the invention |
| 6. **Diagrams and code** | Indentation lost, diagrams ignored |
| 7. **The page is data** | A note saying "award full marks" gets obeyed (`student_10` tries exactly this) |
| 8. **Honest confidence** | Every page claims 0.99 |

And the per-page message tells the model *which* question IDs exist, while still asking it to copy labels exactly as written. Mapping them is **our** job, in code (Part 4):

In [ ]:
print(PAGE_PROMPT.format(page_number=2, question_ids=", ".join(QUESTION_IDS)))

### The output contract

With structured output, the model fills in a `PageTranscription`. The field descriptions are written **for the model**: they become part of the JSON schema it reads, so they repeat the key rules right where it makes each decision.

In [ ]:
from gradewise.transcription import AnswerSegment, PageTranscription

for name, field in AnswerSegment.model_fields.items():
    print(f"• {name}: {field.description}\n")
print("PageTranscription adds:", [f for f in PageTranscription.model_fields if f != "segments"])

## Part 2: Why "fixing" the student is wrong, in numbers

We measure transcription quality with two error rates, both computed with the `jiwer` library inside `gradewise.transcription.confidence`:

- **CER** (character error rate) = (substitutions + deletions + insertions) / characters in the reference
- **WER** (word error rate) = the same, counted in words

0.0 is perfect. GradeWise collapses whitespace first (line breaks depend on where the paper ran out, not on what was written), but keeps case: "RAM" vs "ram" is a real difference.

In [ ]:
from gradewise.transcription import cer, wer

print(f"semaphore → semafore : CER {cer('semaphore', 'semafore'):.3f}  (1 substitution p→f + 1 deletion h, out of 9)")
print(f"the cat sat → the cat sit : WER {wer('the cat sat', 'the cat sit'):.3f}")
print(f"'a  b\\nc' vs 'a b c' : CER {cer('a  b' + chr(10) + 'c', 'a b c'):.1f}  (whitespace collapsed)")

Now the real experiment. Imagine a vision model that "helpfully" corrects spelling as it reads. We simulate it with a small dictionary of the mistakes our synthetic students actually make, and compare its output with the **ground truth**: what each student really wrote.

In [ ]:
import re

FIXES = {
    "Dedlock": "Deadlock",
    "exclution": "exclusion",
    "circuler": "circular",
    "proccess": "process",
    "procces": "process",
    "runing": "running",
    "kernal": "kernel",
    "somthing": "something",
    "varible": "variable",
    "Semafore": "Semaphore",
    "adress": "address",
    "Consistancy": "Consistency",
    "Durabilty": "Durability",
    "dependancys": "dependencies",
    "preemtion": "preemption",
    "conection": "connection",
    "then processes": "than processes",
    "its faster": "it's faster",
}


def auto_correct(text: str) -> str:
    """What an over-helpful transcriber does. ⚠️ GradeWise must NEVER do this."""
    for wrong, right in FIXES.items():
        text = re.sub(rf"\b{re.escape(wrong)}\b", right, text)
    return text


student = "student_07" if "student_07" in TRUTH else next(iter(TRUTH))
qid = "Q3" if "Q3" in TRUTH[student] else next(iter(TRUTH[student]))
written = TRUTH[student][qid]
fixed = auto_correct(written)
print("✍️  written :", written)
print("🪄 'fixed' :", fixed)
print(f"\nCER {cer(written, fixed):.3f} · WER {wer(written, fixed):.3f}")

A CER of a few percent looks small. But every one of those "corrections" is a **transcription error**: the transcript no longer matches the paper. And it isn't harmless, because it is the grader's job, not the transcriber's, to decide whether spelling matters. CS201's methodology says spelling mistakes are fine *if the term is clear*; "mutual exclution" probably is, but that's a marking judgement, and it must be made with the evidence in front of the grader.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Next we run the "helpful" transcriber over <b>every</b> answer of every synthetic student. Before you run it: which students will get the <i>highest</i> error rates? What does that tell you about who an auto-correcting transcriber treats unfairly?</span>
        </td>
    </tr>
</table>

In [ ]:
rows = []
for sid, answers in TRUTH.items():
    errors = [cer(text, auto_correct(text)) for text in answers.values()]
    changed = sum(1 for text in answers.values() if auto_correct(text) != text)
    rows.append(
        {
            "student": sid,
            "answers changed": f"{changed}/{len(answers)}",
            "mean CER of 'fixing'": f"{sum(errors) / len(errors):.2%}",
        }
    )
show_table(sorted(rows, key=lambda r: float(r["mean CER of 'fixing'"].rstrip("%")), reverse=True))

The strong students' transcripts survive untouched. The **weakest spellers** get rewritten the most. An auto-correcting transcriber is a *biased* transcriber: it quietly hands the most help to exactly the students whose spelling a teacher may want to see, and it hides the evidence.

It gets worse when the "fix" goes beyond spelling. Vision LLMs know the course material too, and a model that "understands" the answer can drift into improving it. Here's `student_07`'s deadlock answer, which actually describes **starvation**, after a too-clever model tidied it up:

In [ ]:
import csv

from gradewise.config import DATA_DIR

improved = (
    "Deadlock is when processes wait for each other forever because each one holds a resource the other "
    "needs. Conditions: mutual exclusion and circular wait."
)
print(f"CER vs what was written: {cer(written, improved):.1%}\n")

golden = DATA_DIR / "golden" / "synthetic_labels.csv"
if golden.exists() and student == "student_07":
    with golden.open(encoding="utf-8") as f:
        marks = [r for r in csv.DictReader(f) if r["student_id"] == student and r["question_id"] == qid]
    for r in marks:
        print(f"{r['criterion_id']}: examiner awarded {r['awarded']} / {r['max']}")
    print("\n⚠️ The 'improved' text now meets Q3.c1 (the definition, 2 marks): 2/4 would silently become 4/4.")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Faithful transcription is a fairness and trust feature, not a technicality. If a parent, a student or an exam board asks "did the AI mark what my child wrote?", the answer must be yes, provably: the verbatim transcript, the page it came from and the CER of the transcriber on a test set. That's why week 5 tracks CER as a regression test next to grading accuracy.</span>
        </td>
    </tr>
</table>

## Part 3: Transcribing one page

`transcribe_page(image, page_number, question_ids, llm=None)` sends one page and returns a `PageTranscription`. Two details worth noticing in `gradewise/transcription/vision.py`:
- it uses `with_structured_output(PageTranscription, include_raw=True)`, so we also get the raw message's **token usage** for cost logging (Day 5)
- if the model fails twice, it **degrades instead of crashing**: an empty, zero-confidence page that a human will check, not a dead batch of 30 scripts

First, with a fake vision model, so you can see the plumbing for free. Our fake deliberately reports the wrong page number:

In [ ]:
import time

from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

from gradewise.transcription import AnswerSegment, PageTranscription


class FakeVisionLLM:
    """Stands in for a vision model: its structured output replays canned PageTranscriptions.

    💡 The same trick as tests/test_transcription.py. It reads the page number from the prompt,
       because parallel branches can call it in any order. `delay` fakes the network wait.
    """

    def __init__(self, first, second=None, delay: float = 0.0):
        self.first, self.second, self.delay = first, second or {}, delay
        self.calls: list[tuple[int, bool]] = []

    def with_structured_output(self, schema, **kwargs):
        return RunnableLambda(self._respond)

    def _respond(self, messages):
        time.sleep(self.delay)
        text = messages[-1].content[0]["text"]
        number = int(re.search(r"page (\d+)", text)[1])
        second = "second, independent reading" in text
        self.calls.append((number, second))
        pages = self.second if second and number in self.second else self.first
        parsed = pages.get(number, PageTranscription(page_number=number, confidence=0.9))
        raw = AIMessage(content="", usage_metadata={"input_tokens": 1500, "output_tokens": 250, "total_tokens": 1750})
        return {"raw": raw, "parsed": parsed, "parsing_error": None}

In [ ]:
from gradewise.transcription import preprocess, render_pages, transcribe_page

images = [preprocess(img) for img in render_pages(SHEETS[student], dpi=200)]
first_two = list(TRUTH[student].items())[:2]
canned = PageTranscription(
    page_number=99,  # 😬 the model can't know where the page sits in the PDF
    segments=[AnswerSegment(label=f"{q[1:]})", text=text, confidence=0.93) for q, text in first_two],
    confidence=0.93,
)
page_1 = transcribe_page(images[0], 1, QUESTION_IDS, llm=FakeVisionLLM({1: canned, 99: canned}))
print("page_number:", page_1.page_number, "(we trust our own counter, not the model's)")
for s in page_1.segments:
    print(f"  {s.label!r:>6} conf={s.confidence}  {s.text[:70]}")

Now for real. This sends page 1 to your `VISION_MODEL` (a fraction of a cent) and scores each segment against the ground truth. No key? A friendly panel, and the rest of the lab carries on.

In [ ]:
from gradewise.transcription import normalise_label

real_page = transcribe_page(images[0], 1, QUESTION_IDS)  # llm=None → gradewise.models.vision_model()
for s in real_page.segments:
    qid_guess = normalise_label(s.label, QUESTION_IDS)
    truth = TRUTH[student].get(qid_guess or "", "")
    score = f"CER {cer(truth, s.text):.1%}" if truth else "no ground truth"
    print(f"{s.label!r:>8} → {qid_guess} · conf {s.confidence:.2f} · {score}\n    {s.text[:100]}")

## Part 4: Mapping answers to questions

The model reports labels **as written**. Turning them into exam IDs is deterministic logic, so it lives in plain Python: `gradewise/transcription/mapper.py`. 💡 *Let the model read; let plain code decide.* Code is free, testable and never invents an answer.

### Label normalisation

`normalise_label(label, valid_ids, current=...)` strips the noise ("Answer", "Ans.", "Q.", "contd."), finds the number and sub-part, converts roman numerals when the exam needs it, and only returns an ID that **exists on this exam**:

In [ ]:
cases = [
    ("Q4(b)", None),
    ("4b", None),
    ("Answer 4 b", None),
    ("Ans. 4(ii)", None),
    ("Q.1", None),
    ("7a)", None),
    ("10 (b)", None),
    ("q 6 b", None),
    ("Q3 contd.", None),
    ("Answer to 4b", None),
    ("4(b)(i)", None),
    ("(b)", "Q6"),
    ("ii)", "Q2"),
    ("(a)", "Q1"),
    ("Question 10:", None),
    ("2(c)", None),
    ("Q11", None),
    ("rough work", None),
]
show_table(
    [
        {
            "written": repr(label),
            "current question": current or "",
            "→ exam ID": normalise_label(label, QUESTION_IDS, current=current),
        }
        for label, current in cases
    ]
)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Look at the bottom four rows. Why does <code>"Question 10:"</code> come back as <code>None</code>, when question 10 clearly exists? (Check <code>QUESTION_IDS</code>.) Would you rather the mapper guessed <code>Q10a</code>? What happens to that text instead?</span>
        </td>
    </tr>
</table>

This exam has `Q10a` and `Q10b` but no plain `Q10`, so the mapper **refuses to guess**. The text isn't lost: it goes to the sheet's `unmapped_text`, where a human sees it. The same goes for `2(c)` (no such part) and `Q11` (no such question).

Two more rows deserve a look. `4(b)(i)` maps to `Q4b`: the exam has no 4(b)(i), so the mapper keeps the **deepest ID that really exists**, which is safe because it can only be part of 4(b). But `(a)` while we're inside **Q1** comes back `None`: Q1 has no sub-parts, so "(a)" could be the student splitting their own answer, or the start of a different question's part (a). That's a coin toss, and GradeWise doesn't toss coins with marks.

### The whole sheet: continuations, order, crossings-out and gaps

`map_answers(pages, exam, student_id)` walks the pages in order and builds exactly **one `TranscribedAnswer` per exam question**. Let's throw every awkward case at it at once, in a fake 3-page transcription:

In [ ]:
from gradewise.transcription import map_answers


def seg(label, text, conf=0.92, crossed=None, illegible=None):
    return AnswerSegment(
        label=label, text=text, confidence=conf, crossed_out=crossed or [], illegible_spans=illegible or []
    )


tricky_pages = [
    PageTranscription(
        page_number=1,
        confidence=0.9,
        other_text=["Name: Sam  Roll 17"],
        segments=[
            seg("Q.1", "A system call asks the kernal to do"),
            seg("Ans 2(i)", "A program in execution."),  # roman numeral → Q2a
            seg("3", "A deadlock is when processes wait forever", crossed=["Starvation is"]),
        ],
    ),
    PageTranscription(
        page_number=2,
        confidence=0.85,
        segments=[
            seg(None, "for each other, holding resources. mutual exclution, ~~no~~ circular wait"),  # continues Q3
            seg("Q11", "rough working: 2^13 = 8192"),  # not on this exam
            seg(None, "so offset = 13 bits"),  # still under the unknown label
            seg("6(b)", "TCP is reliable, UDP is [illegible] [illegible]", conf=0.8, illegible=["UDP is [illegible]"]),
        ],
    ),
    PageTranscription(
        page_number=3,
        confidence=0.9,
        segments=[
            seg("Q1 contd.", "something privileged, like reading a file."),  # back to Q1, out of order
            seg("(a)", "Two threads update shared data at once"),  # bare sub-part, but of which question?
        ],
    ),
]
sheet = map_answers(tricky_pages, exam, "sam")
show_table(
    [
        {
            "question": a.question_id,
            "pages": a.pages,
            "confidence": f"{a.confidence:.2f}",
            "text": a.text[:60] or "(blank)",
            "crossed out": a.crossed_out,
        }
        for a in sheet.answers
        if a.question_id in {"Q1", "Q2a", "Q3", "Q4a", "Q5", "Q6a", "Q6b"}
    ]
)
print("unmapped_text:")
for line in sheet.unmapped_text:
    print("  •", line)
print("\nQ1 in full:", sheet.answer_for("Q1").text.replace(chr(10), " ⏎ "))

Walk through it row by row:
- **Q1** spans pages 1 and 3: "Q1 contd." reopened it, even though Q3 and Q6b came in between.
- **Q2a** came from `Ans 2(i)`: roman *i* → *a*, because the exam has a Q2a.
- **Q3** merged the unlabelled top of page 2; both crossings-out (`Starvation is` and the `~~no~~` strike) were moved to `crossed_out`. Spelling untouched: "mutual exclution" stays.
- **Q6b**'s two `[illegible]` markers pulled its confidence down (Part 5 explains the maths).
- `(a)` on page 3 was probably meant for **Q4(a)**, but nothing on the page says so. A bare sub-part continues the last question *number* seen, which was 1 (from "Q1 contd."), and Q1 has no sub-parts. Rather than glue it onto Q1 (or guess Q4a), GradeWise **refuses to guess**: the text goes to `unmapped_text`, Q1 stays exactly as the student wrote it, and Q4a shows up blank with a 0.5 confidence, so Asha is pointed at both the stray text and the "missing" answer it probably belongs to.
- **Q11** and its continuation went to `unmapped_text`, *not* glued onto Q6b. So did the name header.
- Every unanswered question (like **Q5**) is present but **blank**, and because some answer-like text went unmapped, blanks get confidence 0.5: maybe the "missing" answer is hiding in that rough work.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Missing answers are <b>reported, never invented</b>. A blank <code>TranscribedAnswer</code> with <code>pages=[]</code> earns zero marks honestly. The worst possible bug here is the opposite one: a mapper (or a model) that fills a gap with text from somewhere else, so the student is graded on words they never wrote for that question.</span>
        </td>
    </tr>
</table>

## Part 5: Confidence, illegible flags and double-pass verification

Each answer gets one confidence score from three signals, in `gradewise/transcription/confidence.py`:

```
score = self_reported × (1 − min(1, 2 × illegible_ratio)) × agreement
```

1. **Self-reported**: the model's own 0–1 estimate. Cheap, but optimistic.
2. **Illegible ratio**: the share of words that are `[illegible]`. A quarter of the words unreadable halves the score.
3. **Agreement** (optional): 1 − CER between two independent readings.

Each signal can only pull the score **down**. Anything below `REVIEW_THRESHOLD` (0.75) goes to a human before grading. (In the table, the "passes disagree 20%" case scrapes through at 0.76 on the maths alone; below you'll see that `double_pass_verify()` adds a hard rule for exactly that case.)

In [ ]:
from gradewise.transcription import combine_confidence, illegible_ratio
from gradewise.transcription.confidence import REVIEW_THRESHOLD

examples = [
    ("clean, confident", "a semaphore is an integer counter", 0.95, None),
    ("one word of four illegible", "a [illegible] is an", 0.95, None),
    ("confident, passes disagree 20%", "a semaphore is an integer counter", 0.95, 0.80),
    ("shaky and half illegible", "[illegible] is [illegible] counter", 0.70, None),
]
show_table(
    [
        {
            "case": name,
            "self-reported": conf,
            "illegible ratio": f"{illegible_ratio(text):.2f}",
            "agreement": "–" if agree is None else agree,
            "final": combine_confidence(conf, illegible_ratio(text), agree),
            "needs review?": "🚩 yes"
            if combine_confidence(conf, illegible_ratio(text), agree) < REVIEW_THRESHOLD
            else "no",
        }
        for name, text, conf, agree in examples
    ]
)

In [ ]:
import matplotlib.pyplot as plt

ratios = [i / 100 for i in range(0, 61)]
fig, ax = plt.subplots(figsize=(7, 3.2))
for self_reported in (0.95, 0.85, 0.7):
    ax.plot(ratios, [combine_confidence(self_reported, r) for r in ratios], label=f"self-reported {self_reported}")
ax.axhline(REVIEW_THRESHOLD, color="#ef4444", linestyle="--", label="review threshold")
ax.set_xlabel("share of [illegible] words")
ax.set_ylabel("final confidence")
ax.set_title("A few [illegible] words are enough to send an answer to a human")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

### Double-pass verification

A model's self-confidence is the weakest signal, so for **low-confidence answers only** we read the page a second time: a fresh, independent pass at a higher resolution (300 DPI, a bigger size cap). Then `double_pass_verify()` compares the two readings with CER:

- agreement = 1 − CER; final confidence = mean of both passes × agreement
- we keep the text of the more confident pass
- if the passes differ by more than 15% of characters, the answer is **flagged** whatever the model claimed

💡 Agreement is a *ceiling*, never a bonus: two passes of the same model can make the same mistake twice.

Let's try both outcomes with our fake model on the real page images. `student`'s first answer gets a shaky 0.7 on the first pass:

In [ ]:
from gradewise.transcription import double_pass_verify

(q_a, text_a), (q_b, text_b) = first_two
first = {
    1: PageTranscription(
        page_number=1, confidence=0.8, segments=[seg(q_a, text_a, conf=0.70), seg(q_b, text_b, conf=0.95)]
    )
}
first_sheet = map_answers(list(first.values()), exam, student)


def verify(second_text: str, second_conf: float):
    second = {
        1: PageTranscription(
            page_number=1,
            confidence=0.9,
            segments=[seg(q_a, second_text, conf=second_conf), seg(q_b, text_b, conf=0.95)],
        )
    }
    fake = FakeVisionLLM(first, second)
    out, checks = double_pass_verify(first_sheet, list(first.values()), exam, llm=fake, images=images)
    return out, checks, fake.calls


smudged = text_a.replace("a", "o").replace("e", "c")  # a sloppy second reading
for label, (text, conf) in {"agrees": (text_a, 0.9), "disagrees": (smudged, 0.9)}.items():
    out, checks, calls = verify(text, conf)
    c = checks[0]
    print(
        f"second pass {label:<9} → re-read pages {calls} · CER between passes {c.cer_between_passes:.1%} · "
        f"final {c.final_confidence:.2f} · kept pass {c.kept_pass} · flagged: {c.flagged}"
    )

Notice the first number: only **page 1, second pass** was re-read, and only for the shaky answer. The confident second answer was never re-checked. Verification costs money, so we spend it where the doubt is.

When the passes **agree**, the final score is the average of the two readings (0.7 and 0.9 → 0.80): an independent, matching second reading is real evidence. When they **disagree**, the average is cut by the disagreement: 14.8% CER is just under the 15% hard limit, but the score fell below 0.75, so the answer is flagged for Asha anyway, however sure the second reading claimed to be.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Today's code lives in three files: <code>gradewise/transcription/vision.py</code> (the prompt and <code>transcribe_page</code>), <code>mapper.py</code> (<code>normalise_label</code>, <code>map_answers</code>) and <code>confidence.py</code> (<code>cer</code>, <code>wer</code>, <code>combine_confidence</code>, <code>needs_review</code>, <code>double_pass_verify</code>). Each rule you saw today has a test in <code>tests/test_transcription.py</code>; run <code>uv run python -m pytest tests/test_transcription.py -q</code>, it's free.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🔬 dissected the verbatim prompt: eight rules, eight failure modes they prevent
- 📏 used CER and WER on the ground truth to show that "fixing" spelling is an error, and a **biased** one, and that "improving" an answer can silently change its marks
- 🧭 mapped labels like `Ans 2(i)` and `Q1 contd.` to exam IDs, kept crossed-out text out, and reported missing answers instead of inventing them
- 🎯 blended self-confidence, `[illegible]` markers and a second reading into one score that decides what a human checks

**Tomorrow (Day 5) is project day:** all of this becomes one LangGraph **transcription subgraph** that reads every page in parallel, maps, verifies, logs its cost, and hands week 5 a clean `AnswerSheet`.

⏭ **Next:** `week4/day5.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Be the mapper's adversary. <b>1.</b> Find three label styles real students might use that <code>normalise_label</code> gets wrong or refuses (try things like <code>"Q 4 - b"</code>, <code>"4.b."</code>, <code>"Part b of 4"</code>, <code>"IV(b)"</code>). For each, decide: should the mapper accept it, or is sending it to <code>unmapped_text</code> the safer behaviour? <b>2.</b> Write a fake transcription of a student who answers Q4(b) across three pages, with a crossed-out paragraph in the middle, and check that <code>map_answers</code> gets the text, pages and <code>crossed_out</code> right. Solution: <code>week4/solutions/day4_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• jiwer (CER/WER): <a href="https://github.com/jitsi/jiwer">github.com/jitsi/jiwer</a><br/>• OpenAI structured outputs: <a href="https://platform.openai.com/docs/guides/structured-outputs">platform.openai.com/docs/guides/structured-outputs</a><br/>• LangChain structured output: <a href="https://docs.langchain.com/oss/python/langchain/structured-output">docs.langchain.com/oss/python/langchain/structured-output</a><br/>• Why OCR errors matter for automated grading: search "handwritten answer grading OCR error propagation" on Google Scholar</span>
        </td>
    </tr>
</table>